In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import pandas as pd

# Define the path to the CSV file
file_path = '/content/drive/MyDrive/连环画/Ref002/CSV/OCR_Raw_Final.csv'

# Load the CSV file into a pandas DataFrame using the 'python' engine
df = pd.read_csv(file_path, engine='python')

display(df.head())

,Page_Number,ROI,Column,Text_Line,Footnote_Text
0,1,ROI1,Col1,一支驳壳枪,NaN
1,1,ROI1,Col1,出版社：上海人民,NaN
2,1,ROI1,Col1,改编：集体,NaN
3,1,ROI1,Col1,绘画：戴敦邦,NaN
4,1,ROI1,Col1,72年6月1版1次60开,NaN


In [17]:
# Separate main text lines from footnote lines
df_text_lines = df[df['ROI'] != 'FOOTNOTE'].copy()
df_footnotes = df[df['ROI'] == 'FOOTNOTE'].copy()

# Group by Page_Number, ROI, and Column, then concatenate Text_Line for main text
df_processed_text = df_text_lines.groupby(['Page_Number', 'ROI', 'Column'])['Text_Line'].apply(lambda x: '/'.join(x.dropna().astype(str))).reset_index()

# Add an empty 'Footnote_Text' column to df_processed_text for alignment
df_processed_text['Footnote_Text'] = None

# Prepare footnote lines: select relevant columns directly from the original df_footnotes
# The Footnote_Text will naturally be in its own column, and Text_Line will be NaN as in original
df_footnote_rows = df_footnotes[['Page_Number', 'ROI', 'Column', 'Text_Line', 'Footnote_Text']].copy()

# Concatenate the processed text and the original footnote rows
# Ensure column order is consistent before concatenation if necessary, though pandas will align by name
df_final_processed = pd.concat([df_processed_text, df_footnote_rows], ignore_index=True)

# Define a custom order for ROI for sorting
# This order will place FOOTNOTE last among the ROI categories
roi_order = ['ROI1', 'ROI2', 'ROI3', 'ROI4', 'ROI5', 'FOOTNOTE'] # Extend with other ROI values if they exist
df_final_processed['ROI'] = pd.Categorical(df_final_processed['ROI'], categories=roi_order, ordered=True)

# Sort the DataFrame for consistent display:
# Sort by Page_Number, then by the custom ordered ROI, then Column
# Use na_position='last' for Column to put NaN values (from footnotes) at the end
df_final_processed = df_final_processed.sort_values(by=['Page_Number', 'ROI', 'Column'], na_position='last').reset_index(drop=True)

# Display the first few rows of the final processed DataFrame
display(df_final_processed.head(100))

,Page_Number,ROI,Column,Text_Line,Footnote_Text
0,1,ROI1,Col1,一支驳壳枪/出版社：上海人民/改编：集体/绘画：戴敦邦/72年6月1版1次60开/印数：未详...,None
1,1,ROI1,Col2,一张奇怪的药方/出版社：山东人民/改编：刘汉勤/绘画：集体/75年6月1版1次60开/印数：...,None
2,1,ROI1,Col3,一块银元/出版社：人民美术/改编：集体/绘画：集体/72年1月1版1次60开/印数：未详定价...,None
3,1,ROI1,Col4,一颗红心献人民/出版社：人民美术/改编：集体/绘画：集体/72年3月1版1次60开/印数：未...,None
4,1,ROI2,Col1,一块风化石/出版社：广东人民/改编：集体/绘画：集体/72年5月1版1次60开/印数：未详定...,None
...,...,...,...,...,...
95,12,ROI2,Col1,大虎和二虎/出版社：天津人民美术/改编：集体/绘画：华三川/73年2月1版1次60开/印数：...,None
96,12,ROI2,Col2,飞毛腿的故事/出版社：陕西人民/改编：周有恒/绘画：马林/75年1月1版2次60开/印数：4...,None
97,12,ROI2,Col3,大橹的故事/出版社：上海人民/改编：蔡星耀/绘画：罗希贤/74年9月1版1次64开/印数：1...,None
98,12,ROI2,Col4,飞车擒特/出版社：福建人民/改编：郑梦星/绘画：许志棍洪伟阔/75年5月1版1次60开/印数...,None


In [18]:
# Define the path for the new CSV file
output_file_path = '/content/drive/MyDrive/连环画/Ref002/CSV/Ref002_Columns_Grouped.csv'

# Save the processed DataFrame to CSV
df_final_processed.to_csv(output_file_path, index=False)

print(f"DataFrame successfully saved to {output_file_path}")

DataFrame successfully saved to /content/drive/MyDrive/连环画/Ref002/CSV/Ref002_Columns_Grouped.csv
